In [0]:
from pyspark.sql import functions as F, Window

# Ler da tabela Bronze
df_bronze = spark.table("b3_pipeline.bronze_b3_stocks")

# Window particionado por ticker, ordenado por data
w = Window.partitionBy("ticker").orderBy("date")

# Transformações Silver
df_silver = df_bronze \
    .withColumn("date", F.to_date("date")) \
    .withColumn("prev_close", F.lag("close", 1).over(w)) \
    .withColumn("daily_return_pct",
        ((F.col("close") - F.col("prev_close"))
         / F.col("prev_close") * 100)) \
    .withColumn("cumulative_return_pct",
        ((F.col("close") /
          F.first("close").over(w.rowsBetween(
              Window.unboundedPreceding, 0))) - 1) * 100) \
    .filter(F.col("prev_close").isNotNull()) \
    .drop("prev_close")

# Salvar Silver
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("b3_pipeline.silver_b3_stocks")

print("✅ Silver salvo com sucesso!")
df_silver.printSchema()

✅ Silver salvo com sucesso!
root
 |-- date: date (nullable = true)
 |-- close: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- open: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- ticker: string (nullable = true)
 |-- setor: string (nullable = true)
 |-- daily_return_pct: double (nullable = true)
 |-- cumulative_return_pct: double (nullable = true)

